Dataset Loading and Inspection

In [1]:
from sklearn.datasets import fetch_20newsgroups

categories = [
    'sci.space',
    'rec.autos',
    'talk.politics.misc'
]

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

X_train_raw = train_data.data
y_train = train_data.target

X_test_raw = test_data.data
y_test = test_data.target

print("Training Samples:", len(X_train_raw))
print("Testing Samples:", len(X_test_raw))
print("Target Classes:", train_data.target_names)

print("\nSample Training Document:\n")
print(X_train_raw[0][:1000])

Training Samples: 1652
Testing Samples: 1100
Target Classes: ['rec.autos', 'sci.space', 'talk.politics.misc']

Sample Training Document:


   As for advertising -- sure, why not?  A NASA friend and I spent one
   drunken night figuring out just exactly how much gold mylar we'd need
   to put the golden arches of a certain American fast food organization
   on the face of the Moon.  Fortunately, we sobered up in the morning.

Hmmm. It actually isn't all that much, is it? Like about 2 million
km^2 (if you think that sounds like a lot, it's only a few tens of m^2
per burger that said organization sold last year). You'd be best off
with a reflective substance that could be sprayed thinly by an
unmanned craft in lunar orbit (or, rather, a large set of such craft).
If you can get a reasonable albedo it would be visible even at new
moon (since the moon itself is quite dark), and _bright_ at full moon.
You might have to abandon the colour, though.

Buy a cheap launch system, design reusable mo

Text Preprocessing and TF-IDF Feature Extraction

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

max_features = 5000
min_df = 5
max_df = 0.5

tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=max_features,
    min_df=min_df,
    max_df=max_df
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

print("TF-IDF Matrix Shape (Train):", X_train_tfidf.shape)
print("TF-IDF Matrix Shape (Test):", X_test_tfidf.shape)

total_non_zero = X_train_tfidf.nnz
total_elements = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]

sparsity = 1 - (total_non_zero / total_elements)

print("Total Non-Zero Values:", total_non_zero)
print("Sparsity of TF-IDF Matrix:", sparsity)

TF-IDF Matrix Shape (Train): (1652, 4600)
TF-IDF Matrix Shape (Test): (1100, 4600)
Total Non-Zero Values: 83213
Sparsity of TF-IDF Matrix: 0.9890497683966734


Dimensionality Reduction using SVD (LSA)

In [3]:
from sklearn.decomposition import TruncatedSVD

component_values = [50, 100, 200]

svd_models = {}
X_train_lsa = {}
X_test_lsa = {}
variance_results = []

for components in component_values:

    svd_model = TruncatedSVD(
        n_components=components,
        random_state=42
    )

    X_train_reduced = svd_model.fit_transform(X_train_tfidf)
    X_test_reduced = svd_model.transform(X_test_tfidf)

    svd_models[components] = svd_model
    X_train_lsa[components] = X_train_reduced
    X_test_lsa[components] = X_test_reduced

    explained_variance = svd_model.explained_variance_ratio_.sum()
    variance_results.append((components, explained_variance))

    print("\nNumber of Components:", components)
    print("Reduced Training Shape:", X_train_reduced.shape)
    print("Reduced Testing Shape:", X_test_reduced.shape)
    print("Total Explained Variance:", explained_variance)


Number of Components: 50
Reduced Training Shape: (1652, 50)
Reduced Testing Shape: (1100, 50)
Total Explained Variance: 0.1406806645196453

Number of Components: 100
Reduced Training Shape: (1652, 100)
Reduced Testing Shape: (1100, 100)
Total Explained Variance: 0.23116415864264722

Number of Components: 200
Reduced Training Shape: (1652, 200)
Reduced Testing Shape: (1100, 200)
Total Explained Variance: 0.37354302394593153


Normalize Reduced Features

In [4]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()

X_train_lsa_norm = {}
X_test_lsa_norm = {}

for components in component_values:

    X_train_norm = normalizer.fit_transform(X_train_lsa[components])
    X_test_norm = normalizer.transform(X_test_lsa[components])

    X_train_lsa_norm[components] = X_train_norm
    X_test_lsa_norm[components] = X_test_norm

print("Normalization completed for all component settings")

Normalization completed for all component settings


Clustering using K-Means on Normalized LSA Features

In [5]:
from sklearn.cluster import KMeans

kmeans_models = {}
cluster_labels = {}

for components in component_values:

    kmeans = KMeans(
        n_clusters=len(categories),
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_train_lsa_norm[components])

    kmeans_models[components] = kmeans
    cluster_labels[components] = labels

    print("Clustering done for components =", components)

Clustering done for components = 50
Clustering done for components = 100
Clustering done for components = 200


Evaluate Clustering using Silhouette Score

In [6]:
from sklearn.metrics import silhouette_score

silhouette_results = []

for components in component_values:

    score = silhouette_score(
        X_train_lsa_norm[components],
        cluster_labels[components]
    )

    silhouette_results.append((components, score))

    print("Components:", components)
    print("Silhouette Score:", score)
    print("\n")

Components: 50
Silhouette Score: 0.04691952490522305


Components: 100
Silhouette Score: 0.029339975245556402


Components: 200
Silhouette Score: 0.020552891450348523




Classification using Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

classification_results = []

for components in component_values:

    model = LogisticRegression(max_iter=1000)

    model.fit(X_train_lsa_norm[components], y_train)

    predictions = model.predict(X_test_lsa_norm[components])

    accuracy = accuracy_score(y_test, predictions)

    classification_results.append((components, accuracy))

    print("Components:", components)
    print("Classification Accuracy:", accuracy)
    print("-" * 40)

Components: 50
Classification Accuracy: 0.8381818181818181
----------------------------------------
Components: 100
Classification Accuracy: 0.8481818181818181
----------------------------------------
Components: 200
Classification Accuracy: 0.8427272727272728
----------------------------------------


Save Clustering and Classification Results to CSV File

In [8]:
import csv

csv_file_name = "svd_results.csv"

with open(csv_file_name, mode='w', newline='') as file:

    writer = csv.writer(file)

    writer.writerow(["components", "silhouette_score", "accuracy"])

    for i in range(len(silhouette_results)):

      comp = silhouette_results[i][0]
      sil_score = silhouette_results[i][1]
      acc = classification_results[i][1]

      writer.writerow([comp, sil_score, acc])

print("Results saved to svd_results.csv")

Results saved to svd_results.csv


Extract Top Terms per Topic from SVD

In [9]:
import numpy as np

terms = tfidf_vectorizer.get_feature_names_out()

num_topics_to_display = 5
num_top_words = 10

topic_terms_output = {}

for components in component_values:

    svd_model = svd_models[components]
    components_matrix = svd_model.components_

    topics = []

    print("\nTopics for Components =", components, "\n")

    for topic_index in range(num_topics_to_display):

        topic_vector = components_matrix[topic_index]

        top_indices = np.argsort(topic_vector)[::-1][:num_top_words]

        top_terms = [terms[i] for i in top_indices]

        topics.append((topic_index, top_terms))

        print("Topic", topic_index + 1, ":", ", ".join(top_terms))

    topic_terms_output[components] = topics


Topics for Components = 50 

Topic 1 : car, space, people, like, just, don, think, know, new, time
Topic 2 : space, nasa, shuttle, launch, orbit, lunar, moon, station, earth, program
Topic 3 : car, space, cars, engine, thanks, dealer, nasa, shuttle, launch, mail
Topic 4 : men, just, gay, homosexual, edu, sexual, yeah, space, sex, people
Topic 5 : thanks, mail, address, men, car, interested, president, edu, gay, clinton

Topics for Components = 100 

Topic 1 : car, space, people, like, just, don, think, know, new, time
Topic 2 : space, nasa, shuttle, launch, orbit, lunar, moon, station, earth, program
Topic 3 : car, space, cars, engine, thanks, dealer, nasa, shuttle, launch, mail
Topic 4 : men, just, gay, homosexual, edu, sexual, yeah, space, people, sex
Topic 5 : thanks, mail, address, men, car, interested, edu, president, gay, clinton

Topics for Components = 200 

Topic 1 : car, space, people, like, just, don, think, know, new, time
Topic 2 : space, nasa, shuttle, launch, orbit, lun

Save Topic Terms to File

In [10]:
file_name = "lsa_topic_terms.txt"

with open(file_name, "w") as file:

    for components in component_values:

        file.write("Components: " + str(components) + "\n")
        file.write("=" * 50 + "\n")

        topics = topic_terms_output[components]

        for topic_index, terms_list in topics:

            line = "Topic " + str(topic_index + 1) + ": " + ", ".join(terms_list)
            file.write(line + "\n")

        file.write("\n\n")

print("Topic terms saved to lsa_topic_terms.txt")

Topic terms saved to lsa_topic_terms.txt


Display Summary (Variance, Clustering, Classification)

In [11]:
print("\nFINAL SUMMARY\n")

for components, variance in variance_results:

    print("Components:", components)
    print("Explained Variance:", variance)

    for comp, score in silhouette_results:
        if comp == components:
            print("Silhouette Score:", score)

    for comp, acc in classification_results:
        if comp == components:
            print("Classification Accuracy:", acc)

    print("\n")


FINAL SUMMARY

Components: 50
Explained Variance: 0.1406806645196453
Silhouette Score: 0.04691952490522305


Components: 100
Explained Variance: 0.23116415864264722
Silhouette Score: 0.029339975245556402


Components: 200
Explained Variance: 0.37354302394593153
Silhouette Score: 0.020552891450348523




#Conclusion

In this experiment, TF-IDF was used to convert text data from the 20 Newsgroups into a high-dimensional sparse representation. To reduce dimensionality and extract meaningful semantic structure, Truncated SVD (LSA) was applied with different component sizes.

The reduced representations were normalized and evaluated using both clustering and classification techniques. Clustering performance, measured using Silhouette Score, showed that lower dimensions produced more distinct clusters. In contrast, classification accuracy improved with an increase in components up to a certain point, indicating better feature representation, but slightly declined at higher dimensions due to possible noise.

Topic interpretation using top terms confirmed that SVD successfully captured meaningful latent topics from the text data. Overall, the experiment demonstrates that SVD effectively reduces dimensionality, improves computational efficiency, and provides useful representations for both clustering and classification tasks.